In [1]:
# [COLAB SETUP]
import sys
import os

if "google.colab" in sys.modules:
    print("Running in Google Colab. Setting up environment...")
    
    # Mount Google Drive to persist the datasets and cloned repository
    from google.colab import drive
    drive.mount('/content/drive')
    
    repo_path = '/content/drive/MyDrive/Sinhala-Script-Language-Identification-LangID-for-Sinhala-Pali-and-Sanskrit'
    
    if not os.path.exists(repo_path):
        print(f"Cloning repository into {repo_path}...")
        os.makedirs('/content/drive/MyDrive', exist_ok=True)
        os.system(f'git clone https://github.com/Maleesha-K/Sinhala-Script-Language-Identification-LangID-for-Sinhala-Pali-and-Sanskrit.git {repo_path}')
        
    os.chdir(repo_path + '/data_pipeline')
    print("Installing dependencies...")
    os.system('pip install -q pandas scikit-learn fasttext huggingface_hub')
    print("Setup complete!")


In [2]:
input_dir = 'datasets/preprocessed'
output_dir = 'datasets/benchmark_results'


In [3]:
import os
# Auto-resolve the project root if running manually
if not os.path.exists("Makefile") and os.path.exists("../../Makefile"):
    os.chdir("../../")

import json
import glob
import pandas as pd
from sklearn.metrics import accuracy_score, classification_report, f1_score

TARGET_LANGUAGES = {
    "eng": "eng_Latn",
    "sin": "sin_Sinh",

    # Do NOT put "san" here.
    # Sanskrit is handled separately using its script.

    "tam": "tam_Taml",
    "hin": "hin_Deva",
    "ben": "ben_Beng",
    "arb": "arb_Arab",
    "fra": "fra_Latn",
    "deu": "deu_Latn",
}

def detect_script(text):
    """
    Detect the script used by the text.
    """

    text = str(text)

    has_sinhala = any(
        "\u0D80" <= ch <= "\u0DFF"
        for ch in text
    )

    has_devanagari = any(
        "\u0900" <= ch <= "\u097F"
        for ch in text
    )

    if has_sinhala and not has_devanagari:
        return "Sinh"

    if has_devanagari and not has_sinhala:
        return "Deva"

    if has_sinhala and has_devanagari:
        return "Mixed"

    return "Unknown"


def map_true_label(row):

    label = str(row.get("label", "")).strip()
    source = str(row.get("source", "")).strip()

    if label == "san":

        # Sanskrit added by our project in Sinhala script
        if source in [
            "DCS",
            "SansinNT",
            "SiDiaC-v2"
        ]:
            return "san_Sinh"

        # Sanskrit coming from the original benchmark
        return "san_Deva"

    return TARGET_LANGUAGES.get(label)

def load_dataset(file_path):

    print(
        f"\nLoading {os.path.basename(file_path)}..."
    )

    records = []

    with open(file_path, encoding="utf-8") as f:

        for line in f:

            row = json.loads(line)

            raw_label = row.get("label")

            # Keep normal target languages
            # AND keep all Sanskrit examples.
            if (
                raw_label in TARGET_LANGUAGES
                or raw_label == "san"
            ):
                records.append(row)


    df = pd.DataFrame(records)


    if not df.empty:

        # Correct script-aware mapping
        df["flores_label"] = df.apply(
            map_true_label,
            axis=1
        )

        # Only remove Sanskrit rows whose script
        # genuinely could not be determined.
        df = df[
            df["flores_label"].notna()
        ].copy()


        print(
            f"Loaded {len(df)} rows across "
            f"{df['flores_label'].nunique()} "
            f"language-script classes"
        )


        print("\nTrue-label counts:")

        print(
            df["flores_label"]
            .value_counts()
            .sort_index()
        )


    else:

        print(
            "No matching target languages "
            "found in this dataset."
        )


    return df

def evaluate_and_save(results, model_name, dataset_name, target_labels):
    acc = accuracy_score(results["true_label"], results["predicted_label"])
    macro_f1 = f1_score(
        results["true_label"], results["predicted_label"],
        average="macro", labels=target_labels,
    )

    print("\n" + "=" * 48)
    print(f"ZERO-SHOT BENCHMARK RESULTS ({model_name} on {dataset_name})")
    print("=" * 48)
    print(f"Accuracy:  {acc * 100:.2f}%")
    print(f"Macro F1:  {macro_f1 * 100:.2f}%")
    print("=" * 48)
    print("\nPer-language breakdown:\n")
    print(classification_report(
        results["true_label"], results["predicted_label"],
        labels=target_labels, digits=4,
    ))

    os.makedirs(output_dir, exist_ok=True)
    out_file = os.path.join(output_dir, f"{model_name.replace(' ', '_').replace('-', '_').lower()}_{dataset_name}.csv")
    results.to_csv(out_file, index=False)
    print(f"\nSaved predictions to {out_file}\n")
    return results

dataset_files = glob.glob(os.path.join(input_dir, "*.jsonl"))
if not dataset_files:
    print(f"No datasets found in {input_dir}.")


In [4]:
import re
import fasttext
from huggingface_hub import hf_hub_download

print("Downloading OpenLID-v2 model from Hugging Face...")
model_path = hf_hub_download(repo_id="laurievb/OpenLID-v2", filename="model.bin")
print("Loading OpenLID-v2 model...")
model = fasttext.load_model(model_path)
model_name = "OpenLID-v2"
target_labels = [
    "arb_Arab",
    "ben_Beng",
    "deu_Latn",
    "eng_Latn",
    "fra_Latn",
    "hin_Deva",
    "san_Deva",
    "san_Sinh",
    "sin_Sinh",
    "tam_Taml",
]

def clean_for_openlid(text):
    # OpenLID performs best on lowercased text with digits/punctuation stripped.
    text = str(text).strip().replace("\n", " ").lower()
    text = re.sub(r"[^\w\s]|\d", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

for file_path in dataset_files:
    dataset_name = os.path.splitext(os.path.basename(file_path))[0]
    df = load_dataset(file_path)
    if df.empty: continue
    
    texts = df["text"].apply(clean_for_openlid).tolist()
    print(f"Evaluating {len(texts)} samples with {model_name}...")
    preds, _ = model.predict(texts, k=1)

    results = df[["text", "label", "source"]].copy()
    results["true_label"] = df["flores_label"]
    results["predicted_label"] = [p[0].replace("__label__", "") for p in preds]

    evaluate_and_save(results, model_name, dataset_name, target_labels)


Loading OpenLID-v2 model...

Loading commonlid.jsonl...
Loaded 74947 rows across 10 language-script classes

True-label counts:
flores_label
arb_Arab    26152
ben_Beng     1886
deu_Latn     7553
eng_Latn    27461
fra_Latn     3233
hin_Deva     3666
san_Deva      895
san_Sinh     1327
sin_Sinh     2693
tam_Taml       81
Name: count, dtype: int64
Evaluating 74947 samples with OpenLID-v2...

ZERO-SHOT BENCHMARK RESULTS (OpenLID-v2 on commonlid)
Accuracy:  80.30%
Macro F1:  67.17%

Per-language breakdown:



d:\Projects\ML Projects\LangID - DSE project\data_pipeline\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
d:\Projects\ML Projects\LangID - DSE project\data_pipeline\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
d:\Projects\ML Projects\LangID - DSE project\data_pipeline\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this beh

              precision    recall  f1-score   support

    arb_Arab     0.9994    0.9002    0.9473     26152
    ben_Beng     1.0000    0.9459    0.9722      1886
    deu_Latn     0.9956    0.8619    0.9239      7553
    eng_Latn     0.9940    0.8177    0.8973     27461
    fra_Latn     0.9730    0.8463    0.9052      3233
    hin_Deva     0.4974    0.1061    0.1749      3666
    san_Deva     0.7125    0.0637    0.1169       895
    san_Sinh     0.0000    0.0000    0.0000      1327
    sin_Sinh     0.6647    0.9770    0.7912      2693
    tam_Taml     0.9877    0.9877    0.9877        81

   micro avg     0.9679    0.8030    0.8778     74947
   macro avg     0.7824    0.6507    0.6717     74947
weighted avg     0.9382    0.8030    0.8554     74947


Saved predictions to datasets/benchmark_results\openlid_v2_commonlid.csv


Loading flores_plus.jsonl...
Loaded 13128 rows across 10 language-script classes

True-label counts:
flores_label
arb_Arab    2024
ben_Beng    1012
deu_Latn    1012


d:\Projects\ML Projects\LangID - DSE project\data_pipeline\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
d:\Projects\ML Projects\LangID - DSE project\data_pipeline\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
d:\Projects\ML Projects\LangID - DSE project\data_pipeline\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this beh


Saved predictions to datasets/benchmark_results\openlid_v2_flores_plus.csv


Loading wili-2018.jsonl...
Loaded 11020 rows across 9 language-script classes

True-label counts:
flores_label
ben_Beng    1000
deu_Latn    1000
eng_Latn    1000
fra_Latn    1000
hin_Deva    1000
san_Deva    1000
san_Sinh    1327
sin_Sinh    2693
tam_Taml    1000
Name: count, dtype: int64
Evaluating 11020 samples with OpenLID-v2...

ZERO-SHOT BENCHMARK RESULTS (OpenLID-v2 on wili-2018)
Accuracy:  67.41%
Macro F1:  56.62%

Per-language breakdown:

              precision    recall  f1-score   support

    arb_Arab     0.0000    0.0000    0.0000         0
    ben_Beng     1.0000    0.8910    0.9424      1000
    deu_Latn     0.9990    0.9650    0.9817      1000
    eng_Latn     0.9015    0.9880    0.9427      1000
    fra_Latn     0.9947    0.9440    0.9687      1000
    hin_Deva     0.0410    0.0080    0.0134      1000
    san_Deva     1.0000    0.0140    0.0276      1000
    san_Sinh     0.0000    0.0000    0

d:\Projects\ML Projects\LangID - DSE project\data_pipeline\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
d:\Projects\ML Projects\LangID - DSE project\data_pipeline\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
d:\Projects\ML Projects\LangID - DSE project\data_pipeline\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
 